In [ ]:
# Default parameter values — overridden by papermill at runtime
run_id = "00000000-0000-0000-0000-000000000000"
bucket = "django-prefect-datalake-dev"
aws_access_key_id = "rustfs"
aws_secret_access_key = "rustfs_secret"
aws_s3_region = "us-east-1"
s3_endpoint = "localhost:9000"
notebook_output_dir = "data/notebook_outputs"

In [ ]:
import json
import s3fs
import polars as pl

In [ ]:
endpoint_url = f"http://{s3_endpoint}" if not s3_endpoint.startswith("http") else s3_endpoint
storage_options = {
    "key": aws_access_key_id,
    "secret": aws_secret_access_key,
    "endpoint_url": endpoint_url,
}
s3 = s3fs.S3FileSystem(
    key=aws_access_key_id,
    secret=aws_secret_access_key,
    endpoint_url=endpoint_url,
    client_kwargs={"region_name": aws_s3_region},
)

In [ ]:
input_path = f"s3://{bucket}/processed/flows/credit-prediction/{run_id}/predict_01_raw.parquet"
df = pl.read_parquet(input_path, storage_options=storage_options)
row = df.row(0, named=True)

In [ ]:
income = float(row["income"])
age = float(row["age"])
credit_score = float(row["credit_score"])
employment_years = float(row["employment_years"])

# Normalise each factor to [0, 1]
credit_score_norm = max(0.0, min(1.0, (credit_score - 300) / 550))  # FICO 300–850
income_norm       = max(0.0, min(1.0, income / 150_000))             # up to $150k
employment_norm   = max(0.0, min(1.0, employment_years / 20))        # up to 20 yrs
age_norm          = 1.0 if 22 <= age <= 70 else 0.6                  # optimal age band

# Weighted score
score = (
    credit_score_norm * 0.40
    + income_norm     * 0.30
    + employment_norm * 0.20
    + age_norm        * 0.10
)
score = round(score, 4)
confidence = round(score * 100, 1)

if score >= 0.70:
    classification = "Approved"
elif score >= 0.50:
    classification = "Review"
else:
    classification = "Declined"

print(f"Score: {score}  Classification: {classification}  Confidence: {confidence}%")

In [ ]:
result_df = pl.DataFrame({
    "income":           [income],
    "age":              [age],
    "credit_score":     [credit_score],
    "employment_years": [employment_years],
    "score":            [score],
    "classification":   [classification],
    "confidence":       [confidence],
})

s3_output_path = f"processed/flows/credit-prediction/{run_id}/output.parquet"
output_full = f"s3://{bucket}/{s3_output_path}"

with s3.open(output_full, "wb") as f:
    result_df.write_parquet(f, compression="snappy", use_pyarrow=True)

print(f"Scored → {output_full}")

In [ ]:
# IMPORTANT: This JSON is parsed by PipelineRunner._extract_metadata()
# Must be the last printed line and valid JSON.
print(json.dumps({
    "row_count":      1,
    "s3_output_path": s3_output_path,
    "score":          score,
    "classification": classification,
    "confidence":     confidence,
}))